In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

file_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(file_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=40, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
#Drop Order_ID column
df.drop(columns=['Order_ID'],inplace=True)


In [ ]:
# Task 2: Write your code here:
#Handling missing values
for col in df.columns:
  if df[col].isnull().sum != 0 and df[col].dtypes == 'float64':
    df[col].fillna(df[col].mean(),inplace=True)

df.dropna(inplace=True)


In [ ]:
# Task 3: Write your code here:
#Removing Duplicates
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    df.drop_duplicates(inplace=True)

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
#Encode the catogical columns
from sklearn.preprocessing import LabelEncoder


for col in df.columns:
  if df[col].dtypes == 'object': # Checking the category columns
    label_encoder = LabelEncoder()
    df[col] = label_encoder.fit_transform(df[col])
df

In [ ]:
# # Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df = scaler.fit_transform(df)



In [ ]:
# Task 6: Write your code here:
#the data is balanced

In [ ]:
# Task 1: Write your code here:
#Split dataset
X = df.drop(columns=['Delivery_Time'],inplace=False)
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


#Score for each Fold
scores = 0

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train the Model
    model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    scores += mae

print(f"Averaged score across all folds: {scores/5}")

In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
#

In [ ]:
# Task Bonus: Write your code here:

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Train the Model
    model_forest = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model_forest.fit(X_train, y_train)

    model_cat = CatBoostRegressor(verbose=0)
    model_cat.fit(X_train, y_train)

    # Predict and evaluate
    y_pred_forest = model_forest.predict(X_test)
    y_pred_cat = model_forest.predict(X_test)

    y_pred_average = (y_pred_forest + y_pred_cat)/2

    mae = mean_absolute_error(y_test, y_pred_average)

    print("MAE:",mae)